# Notebook 02_02 — Feature Selection: ChiSquare

Runs **ChiSquare** across K = {4, 8, 16} × 6 classifiers × 5 seeds.
Uses the pre-computed ranking from `02_00_fs_rankings.ipynb` (`models/fs_rankings.pkl`).

**Results saved incrementally to** `results/fs_chisquare_raw.csv` — if this notebook crashes mid-run, just re-run it: completed (K, seed, classifier) combinations are skipped automatically.

In [1]:
import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CODE_DIR = '/content/UAV' if IN_COLAB else os.environ.get('UAV_CODE_DIR', 'D:/UAV_')

if IN_COLAB and not os.path.isdir(os.path.join(CODE_DIR, '.git')):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/haribawngg/UAV.git', CODE_DIR], check=True)

sys.path.insert(0, os.path.join(CODE_DIR, 'notebook'))
from common import *

print('Imports OK')

CPU: 16 logical cores detected -> N_JOBS=-1 (RF/XGB/LGBM/KNN use all cores)
Imports OK


In [2]:
d = load_data()
X_train, X_val, X_test = d['X_train'], d['X_val'], d['X_test']
y_train = d['y_train']
FEATURE_NAMES = d['feature_names']

with open(f'{MODELS_DIR}fs_rankings.pkl', 'rb') as f:
    RANKINGS = pickle.load(f)

METHOD_NAME = 'ChiSquare'
ranked_indices = RANKINGS[METHOD_NAME]

print(f'Method: {METHOD_NAME}')
print('Top-16 features:', [FEATURE_NAMES[i] for i in ranked_indices[:16]])

Method: ChiSquare
Top-16 features: ['frame.encap_type', 'wlan_radio.frequency', 'radiotap.channel.freq', 'wlan_radio.phy', 'radiotap.length', 'wlan_radio.timestamp', 'radiotap.mactime', 'wlan_radio.end_tsf', 'wlan_radio.start_tsf', 'wlan.fc.subtype', 'radiotap.channel.flags.ofdm', 'data.len', 'arp.opcode', 'wlan.fc.type', 'tcp.flags.push', 'tcp.time_relative']


## Transform function (slice to top-K features by this method's ranking)

In [3]:
def transform_fn(K):
    selected = ranked_indices[:K]
    return X_train[:, selected], X_val[:, selected], X_test[:, selected], K

## Run experiment grid (resumable)

In [4]:
RESULTS_CSV = f'{RESULTS_DIR}fs_chisquare_raw.csv'

run_experiment_grid(
    method_name=METHOD_NAME,
    transform_fn=transform_fn,
    d=d,
    results_csv_path=RESULTS_CSV
)

[ChiSquare] resuming: 10/90 combinations already done.
[ChiSquare] K=4 seed=52 KNN    F1=0.1157  train=0.1s  inf=0.9477ms/sample  (11/90)
[ChiSquare] K=4 seed=52 MLP    F1=0.2174  train=17.7s  inf=0.0003ms/sample  (12/90)
[ChiSquare] K=4 seed=62 DT     F1=0.2244  train=0.0s  inf=0.0000ms/sample  (13/90)
[ChiSquare] K=4 seed=62 RF     F1=0.3089  train=1.0s  inf=0.0027ms/sample  (14/90)
[ChiSquare] K=4 seed=62 XGB    F1=0.2244  train=1.2s  inf=0.0006ms/sample  (15/90)
[ChiSquare] K=4 seed=62 LGBM   F1=0.2232  train=0.6s  inf=0.0007ms/sample  (16/90)
[ChiSquare] K=4 seed=62 KNN    F1=0.1157  train=0.1s  inf=1.4348ms/sample  (17/90)
[ChiSquare] K=4 seed=62 MLP    F1=0.2244  train=16.6s  inf=0.0010ms/sample  (18/90)
[ChiSquare] K=4 seed=72 DT     F1=0.2244  train=0.0s  inf=0.0000ms/sample  (19/90)
[ChiSquare] K=4 seed=72 RF     F1=0.3089  train=1.0s  inf=0.0027ms/sample  (20/90)
[ChiSquare] K=4 seed=72 XGB    F1=0.2244  train=1.1s  inf=0.0005ms/sample  (21/90)
[ChiSquare] K=4 seed=72 LGBM  

## Quick summary

In [5]:
res_df = pd.read_csv(RESULTS_CSV)
summary = res_df.groupby(['K', 'classifier'])[['f1', 'train_time_s', 'inference_ms']].agg(['mean', 'std'])
print(summary.to_string())

                     f1           train_time_s           inference_ms              
                   mean       std         mean       std         mean           std
K  classifier                                                                      
4  DT          0.224437  0.000005     0.013031  0.000655     0.000043  1.784866e-06
   KNN         0.115686  0.000002     0.071840  0.002359     1.302484  2.145301e-01
   LGBM        0.222784  0.003064     0.556899  0.018520     0.000693  1.854243e-05
   MLP         0.221612  0.003863    16.689879  2.356378     0.000725  5.749294e-04
   RF          0.256569  0.047858     0.997398  0.004054     0.002668  3.973018e-05
   XGB         0.224439  0.000007     1.179423  0.093496     0.000566  1.988728e-05
8  DT          0.309030  0.000000     0.038864  0.002349     0.000041  9.244330e-07
   KNN         0.309030  0.000000     0.134665  0.005602     1.339665  1.723308e-01
   LGBM        0.309030  0.000000     0.612314  0.014992     0.000625  3.584